# FASE 5: SKOR RISIKO GABUNGAN
Di tahap akhir ini, kita akan menggabungkan semua faktor bahaya (panasnya api, jarak ke sekolah, kepadatan penduduk, dan efek musim kemarau) menjadi satu Skor Risiko (0 sampai 100).
Semakin tinggi skornya, semakin mendesak titik api tersebut untuk ditangani.

In [ ]:
from IPython.display import display, Markdown
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import os
import time

start_time = time.time()
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

fig_dir = '../outputs/figures'
sns.set_theme(style="whitegrid")
PALETTE_RED = "#C0392B"
PALETTE_GOLD = "#F39C12"

## 1. Pembuatan Skor Risiko
Rumusnya sederhana: tingkat bahaya alamiah (FRP) digabung dengan seberapa dekat api ke sekolah, lalu dikalikan dengan musim kemarau (kalau lagi kemarau, skornya makin tinggi).
Kita juga membagi level risikonya jadi Rendah, Sedang, Tinggi, dan Kritis.

In [ ]:
df_master = pd.read_csv('../data/processed/hotspot_master.csv')

frp_cap = df_master['frp'].quantile(0.99)
df_master['frp_capped'] = df_master['frp'].clip(upper=frp_cap)
frp_score = (df_master['frp_capped'] / frp_cap) * 100

dist_school = df_master['dist_nearest_school_km'].fillna(100)
df_master['exposure_score'] = np.maximum(0, 100 * (1 - (dist_school / 10)))

# Population multiplier dihilangkan sementara dari formula karena resolusi provinsi terlalu kasar
# pop = df_master['population'].fillna(df_master['population'].mean())
# pop_mult = 0.8 + 0.4 * ((pop - min_pop) / (max_pop - min_pop))

season_mult = df_master['is_dry_season'].apply(lambda x: 1.5 if x else 1.0)

# -----------------------------------------------------------------------
# CATATAN METODOLOGIS — COMPOSITE RISK SCORE
# -----------------------------------------------------------------------
# Formula di bawah menggunakan asumsi desain yang BELUM divalidasi secara empiris:
#   - Bobot 60% FRP + 40% exposure_score: dipilih berdasarkan judgment domain,
#     bukan kalibrasi terhadap ground-truth kejadian kebakaran nyata.
#   - Pengganda musim kemarau 1.5x: asumsi bahwa musim kemarau meningkatkan risiko
#     50% secara rata-rata, tanpa validasi data historis kebakaran.
#   - Radius 10 km di exposure_score: dipilih berdasarkan distribusi empiris
#     (P96 jarak hotspot ke sekolah), namun belum ada bukti bahwa 10 km adalah
#     batas dampak asap yang signifikan secara kesehatan untuk wilayah Kalimantan.
# Output skor ini adalah SKOR PRIORITISASI EKSPLORATIF, bukan penilaian risiko
# operasional. Distribusi kategori (% Rendah/Sedang/Tinggi/Kritis) bergantung
# sepenuhnya pada threshold dan formula di atas.
# -----------------------------------------------------------------------

base_score = (0.6 * frp_score) + (0.4 * df_master['exposure_score'])
final_score = base_score * season_mult

from scipy.stats import rankdata

df_master['_raw_score'] = final_score
df_master['is_kritis_abs'] = (df_master['frp_tier'] == 'Ekstrem') | (df_master['dist_nearest_school_km'] <= 5)

def assign_non_kritis_cat(score, thresholds):
    if score >= thresholds[1]: return 'Tinggi'
    elif score >= thresholds[0]: return 'Sedang'
    else: return 'Rendah'

if (~df_master['is_kritis_abs']).sum() > 0:
    non_kritis_scores = df_master.loc[~df_master['is_kritis_abs'], '_raw_score']
    t_33 = np.percentile(non_kritis_scores, 33.33)
    t_66 = np.percentile(non_kritis_scores, 66.66)
    df_master.loc[~df_master['is_kritis_abs'], 'risk_category'] = non_kritis_scores.apply(lambda x: assign_non_kritis_cat(x, [t_33, t_66]))

df_master.loc[df_master['is_kritis_abs'], 'risk_category'] = 'Kritis'

cat_bases = {'Rendah': 0, 'Sedang': 25, 'Tinggi': 50, 'Kritis': 75}
df_master['risk_score'] = 0.0

for cat, base in cat_bases.items():
    mask = df_master['risk_category'] == cat
    if mask.sum() > 0:
        ranks = rankdata(df_master.loc[mask, '_raw_score'])
        pct = (ranks - 1) / len(ranks)
        df_master.loc[mask, 'risk_score'] = base + (pct * 25)

df_master['risk_score'] = df_master['risk_score'].round(1)
df_master['is_peat_proxy'] = (df_master['daynight'] == 'N') & (df_master['frp_tier'].isin(['Rendah', 'Menengah']))
print("     Distribusi Tingkatan Ancaman:")
print(df_master['risk_category'].value_counts())

# -----------------------------------------------------------------------
# CONFIDENCE SEBAGAI INDIKATOR KUALITAS OBSERVASI (bukan pengali risiko)
# -----------------------------------------------------------------------
# NASA FIRMS mendokumentasikan bahwa confidence VIIRS membantu menilai kualitas
# pixel (faktor seperti ukuran piksel, keberadaan awan, dan jenis permukaan).
# Tidak ada cutoff optimal yang berlaku universal, dan confidence rendah TIDAK
# berarti api separuh berbahaya (bisa jadi piksel besar atau tutupan awan parsial).
# Oleh karena itu, confidence ditampilkan sebagai label kualitas observasi terpisah,
# BUKAN sebagai pengali numerik yang mengurangi skor risiko.
# Referensi: NASA FIRMS VIIRS Active Fire User Guide.
# -----------------------------------------------------------------------
conf_quality_map = {'h': 'Tinggi', 'n': 'Nominal', 'l': 'Rendah'}
if 'confidence' in df_master.columns:
    df_master['confidence_quality'] = df_master['confidence'].map(conf_quality_map).fillna('Nominal')
else:
    df_master['confidence_quality'] = 'Nominal'

# verification_priority = risk_score (tanpa pengali confidence, sesuai panduan NASA FIRMS)
df_master['verification_priority'] = df_master['risk_score']
df_master['verification_priority'] = df_master['verification_priority'].round(1)

## 2. Visualisasi Skor Risiko
Mari kita lihat ada berapa titik api yang masuk ke level Rendah, Sedang, Tinggi, atau Kritis. 
Kita juga akan mengecek provinsi mana yang rata-rata skor risikonya paling mengkhawatirkan.

In [ ]:
cat_colors = {'Rendah': '#3498DB', 'Sedang': '#F39C12', 'Tinggi': '#E67E22', 'Kritis': '#C0392B'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
bins = np.arange(0, 105, 5)
for cat, color in cat_colors.items():
    subset = df_master[df_master['risk_category'] == cat]['risk_score']
    ax.hist(subset, bins=bins, color=color, alpha=0.85, edgecolor='white', linewidth=0.4, label=cat)

ax.set_title('Distribusi Skor Risiko Komposit\nWarna menunjukkan kategori bahaya titik api', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Skor Risiko (0 = Aman, 100 = Sangat Berbahaya)')
ax.set_ylabel('Jumlah Titik Panas')
ax.legend(title='Kategori Risiko')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

ax = axes[1]
cat_order = ['Rendah', 'Sedang', 'Tinggi', 'Kritis']
cat_counts = df_master['risk_category'].value_counts().reindex(cat_order, fill_value=0)
cat_pct = cat_counts / cat_counts.sum() * 100

bars = ax.barh(cat_order, cat_counts.values,
               color=[cat_colors[c] for c in cat_order], edgecolor='white')
for i, (v, pct) in enumerate(zip(cat_counts.values, cat_pct.values)):
    ax.text(v + max(cat_counts)*0.01, i, f"{v:,}  ({pct:.1f}%)", va='center', fontsize=10)

ax.set_title('Jumlah Titik Api per Kategori Risiko\nDari Rendah hingga Kritis', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Jumlah Titik Panas')
ax.set_ylabel('Kategori Risiko')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('Hasil Penilaian Risiko Komposit FIRELINE', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{fig_dir}/E1_distribusi_skor_risiko.png', dpi=150, bbox_inches='tight')
plt.show()

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
prov_risk = df_master.groupby('province_name')['risk_score'].agg(['mean', 'max', 'count']).sort_values('mean', ascending=False)

x = range(len(prov_risk))
bars = ax.bar(x, prov_risk['mean'], color=PALETTE_RED, alpha=0.8, label='Rata-rata skor risiko', edgecolor='white')
ax.scatter(x, prov_risk['max'], color='#2C3E50', zorder=5, s=60, label='Skor risiko tertinggi', marker='D')

ax.set_xticks(x)
ax.set_xticklabels(prov_risk.index, rotation=15, ha='right')
ax.set_title('Rata-rata dan Skor Tertinggi Risiko per Provinsi\nDiamond = skor puncak, Batang = rata-rata keseluruhan', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Provinsi')
ax.set_ylabel('Skor Risiko (0-100)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{fig_dir}/E2_risiko_provinsi.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Simpan Data Matang
Data yang sudah punya skor risiko ini akan kita simpan sebagai data pamungkas yang nanti dipakai di dashboard.

In [ ]:
df_master.to_csv('../data/processed/hotspot_scored_final.csv', index=False)
display(Markdown("**Data final dengan skor risiko berhasil disimpan!** (`hotspot_scored_final.csv`)"))

## 4. Jawaban 4 Pertanyaan Utama Proyek
Kita kembali ke awal. Proyek ini harus bisa menjawab 4 pertanyaan mendesak. 
Berikut adalah daftarnya berdasarkan data riil yang sudah kita hitung.

In [ ]:
top_urgent = df_master.sort_values(by='risk_score', ascending=False).head(5)
top_vuln = df_master.sort_values(by=['exposure_score', 'population'], ascending=[False, False]).head(5)
top_verify = df_master.sort_values(by='verification_priority', ascending=False).head(5)

display(Markdown("**Tabel Top 5 Titik Paling Kritis (Berdasarkan Skor Risiko):**"))
display(top_urgent[['date_local', 'province_name', 'frp', 'dist_nearest_school_km', 'risk_score']])

with open('../outputs/core_questions_answers.md', 'w', encoding='utf-8') as f:
    f.write("# Jawaban 4 Pertanyaan Operasional Utama FIRELINE\n\n")

    f.write("## 1. Area mana yang paling mendesak untuk diselamatkan saat ini?\n")
    f.write("Berdasarkan Skor Risiko Komposit tertinggi (gabungan intensitas, paparan, dan populasi):\n")
    for i, row in top_urgent.iterrows():
        f.write(f"- Titik di {row.get('province_name', 'Tidak diketahui')} (Lat: {row['latitude']:.4f}, Lon: {row['longitude']:.4f}) dengan Skor Kritis: **{row['risk_score']}** (FRP: {row['frp']} MW)\n")

    f.write("\n## 2. Populasi mana yang paling rentan?\n")
    f.write("Berdasarkan kedekatan titik api ke lokasi Sekolah dan kepadatan populasi provinsi:\n")
    for i, row in top_vuln.iterrows():
        f.write(f"- Area {row.get('province_name', 'Tidak diketahui')} (Lat: {row['latitude']:.4f}) memiliki kebakaran berjarak hanya **{row.get('dist_nearest_school_km', 100):.2f} km** dari sekolah terdekat.\n")

    f.write("\n## 3. Laporan mana yang harus diverifikasi terlebih dahulu?\n")
    f.write("Berdasarkan Prioritas Verifikasi (Skor Risiko tertimbang oleh kepercayaan satelit):\n")
    for i, row in top_verify.iterrows():
        f.write(f"- Titik di {row.get('province_name', 'Tidak diketahui')} (Kepercayaan: {row.get('confidence', 'n')}) dengan Skor Prioritas: **{row['verification_priority']}**\n")

    f.write("\n## 4. Respons apa yang harus diprioritaskan terlebih dahulu?\n")
    kritis_via_frp = ((df_master['risk_category'] == 'Kritis') & (df_master['frp_tier'] == 'Ekstrem')).sum()
    kritis_via_sekolah = ((df_master['risk_category'] == 'Kritis') & (df_master['dist_nearest_school_km'] <= 5)).sum()
    kritis_count = (df_master['risk_category'] == 'Kritis').sum()
    
    f.write(f"- Total titik dengan kategori Kritis secara absolut: **{kritis_count:,}** titik.\n")
    f.write(f"  *(Catatan: Distribusi kategori bergantung pada formula dan threshold asumsi di atas — hasilnya adalah skor prioritisasi eksploratif, bukan penilaian risiko operasional tervalidasi.)*\n")
    f.write(f"- 🚁 Kandidat Prioritas Verifikasi Udara (Kritis via FRP Ekstrem > P95): **{kritis_via_frp:,}** titik yang perlu pengecekan lapangan. FRP tinggi mengindikasikan intensitas termal, namun tidak secara otomatis membutuhkan water bombing tanpa konfirmasi lebih lanjut.\n")
    f.write(f"- 🚒 Kandidat Prioritas Assessment Dampak Sosial (Kritis via Jarak Sekolah ≤ 5km): **{kritis_via_sekolah:,}** titik yang perlu assessment kedekatan ke komunitas. Kedekatan geografis ke koordinat sekolah belum berarti sekolah aktif atau warga sedang terancam — perlu verifikasi lapangan.\n")
    f.write("> **Catatan:** Titik jalur udara dan jalur darat di atas bisa saja tumpang tindih (overlap) jika ada api ekstrem yang kebetulan sangat dekat dengan sekolah.\n")

with open('../outputs/core_questions_answers.md', 'r', encoding='utf-8') as f:
    laporan_akhir = f.read()

display(Markdown(laporan_akhir))